# Question 1
 Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table. 

In [0]:
%sql
create schema if not exists cyntexa_dev.Day_8;
create volume if not exists cyntexa_dev.Day_8.my_volume;

In [0]:
%sql
desc  dev.bronze.customers_raw;

In [0]:
%sql


CREATE TABLE IF NOT EXISTS cyntexa_dev.day_8.customers_merge (
    customer_id INT,
    name STRING,
    email STRING,
    city STRING,
    state STRING,
    signup_date DATE,
    phone BIGINT
)
USING DELTA;

In [0]:
%sql
merge with schema evolution  into cyntexa_dev.day_8.customers_merge  t
using  (
    select * from dev.bronze.customers_raw s
)
on t.customer_id = s.customer_id
when matched then update set *
when not matched then insert *;
    


Delta Lake Upsert (MERGE INTO) with Schema Evolution
Overview
This workflow demonstrates a standard SCD Type 1 (Slowly Changing Dimension) pattern in Databricks Delta Lake. It synthesizes incoming raw batch data (dev.bronze.customers_raw) into a target Silver layer table (cyntexa_dev.day_8.customers_merge).

Upsert Operation: Performs concurrent UPDATE (for existing key matches) and INSERT (for new records).

Automatic Schema Evolution: Dynamically alters the target table structure if new columns are introduced in the source dataset.  

# Question 2

In [0]:
%sql
create  or replace view cyntexa_dev.day_8.customers_merge_view_masked as
select customer_id , name,
concat(left(email,3),'***@***.com') as masked_email ,
city,
state,
signup_date,
concat('XXXXXX', right(phone,3)) as masked_phone

  from cyntexa_dev.day_8.customers_merge

In [0]:
%sql
select * from cyntexa_dev.day_8.customers_merge_view_masked;

In [0]:
%sql
-- 1. Groups banao
CREATE GROUP full_access_group;
CREATE GROUP restricted_group;


ALTER GROUP full_access_group ADD USER `user1@example.com`;
ALTER GROUP restricted_group ADD USER `user2@example.com`;



GRANT USE CATALOG ON CATALOG cyntexa_dev TO `full_access_group`;
GRANT USE SCHEMA ON SCHEMA cyntexa_dev.day_8 TO `full_access_group`;
GRANT SELECT ON TABLE cyntexa_dev.day_8.customers_merge TO `full_access_group`;


GRANT USE CATALOG ON CATALOG cyntexa_dev TO `restricted_group`;
GRANT USE SCHEMA ON SCHEMA cyntexa_dev.day_8 TO `restricted_group`;
GRANT SELECT ON VIEW cyntexa_dev.day_8.customers_merge_view_masked TO `restricted_group`;



    

# Question 3

Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for. 

A **DBU (Databricks Unit)** is a normalized unit of processing capacity per hour, billed down to the second.

Think of a DBU like a **taximeter for software processing capacity**. It abstracts away the complex math of how many CPU cores, RAM gigabytes, or GPUs your cluster is using into a single, unified currency.

### What a DBU is Billing For

When you run a compute resource on Databricks, your bill is divided into two distinct parts:

1. **Cloud Infrastructure (AWS/Azure/GCP):** You pay your cloud provider directly for renting the raw hardware (the underlying virtual machines, RAM, disks, and networking).
2. **Databricks Units (DBUs):** You pay Databricks for the software layer—the managed Spark engine, query optimization algorithms, workspace features, and platform orchestration running on top of those machines.

### The 4 Variables That Control Your DBU Rate

You don't pay a fixed price per hour of machine time; the number of DBUs consumed per hour depends on four factors:

* **Hardware Power:** Larger instances with more CPU cores and RAM burn DBUs faster than smaller instances.
* **Workload Type:** Running an interactive workspace (All-Purpose Compute for ad-hoc coding) costs significantly more DBUs per hour than running a scheduled background task (Jobs Compute), even on the exact same hardware.
* **Feature Enhancements:** Turning on acceleration engines like **Photon** increases the DBU consumption rate per hour because you are using higher-tier software optimization.
* **Serverless vs. Classic:** In classic clusters, DBUs only cover software (and cloud costs are billed separately). In Serverless workloads, the DBU rate is higher because it bundles both the software layer and the raw cloud infrastructure into a single DBU bill.

### Summary Formula

$$\text{Total DBU Cost} = \text{Time (Hours)} \times \text{DBU Consumption Rate} \times \text{Dollar Price per DBU}$$

* **Time:** Exact seconds your cluster is active.
* **DBU Rate:** How hard the software and hardware are working (determined by instance size and cluster type).
* **Price:** Your negotiated or list price rate for a single DBU (e.g., ~$0.15–$0.55/DBU depending on your tier and workload type).